# DFM Specification: Building State Space Models

**Goal:** Convert lag polynomial DFM to state space form.

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dfm_to_statespace(Lambda_list, Phi_list, Q, H):
    """
    Convert DFM to state space form.
    
    Lambda_list: [Λ₀, Λ₁, ..., Λₚ]
    Phi_list: [Φ₁, ..., Φ_q]
    """
    N, r = Lambda_list[0].shape
    p = len(Lambda_list) - 1
    q = len(Phi_list)
    s = max(p + 1, q)
    
    # Z matrix
    Z = np.zeros((N, r * s))
    for i, L in enumerate(Lambda_list):
        Z[:, i*r:(i+1)*r] = L
    
    # T matrix (companion form)
    T = np.zeros((r * s, r * s))
    for i, Phi in enumerate(Phi_list):
        T[:r, i*r:(i+1)*r] = Phi
    if s > 1:
        T[r:, :-r] = np.eye(r * (s - 1))
    
    # R matrix
    R = np.zeros((r * s, r))
    R[:r, :] = np.eye(r)
    
    return Z, T, R

# Example: 2 factors, 1 lag in observations, AR(1) factors
N, r = 10, 2
Lambda_0 = np.random.randn(N, r)
Lambda_1 = np.random.randn(N, r) * 0.3
Phi_1 = np.diag([0.8, 0.6])
Q = np.eye(r) * 0.5
H = np.eye(N) * 0.3

Z, T, R = dfm_to_statespace([Lambda_0, Lambda_1], [Phi_1], Q, H)

print(f"State dimension: {T.shape[0]}")
print(f"Observation dimension: {Z.shape[0]}")
print(f"\nT matrix:\n{T}")
print(f"\nMax eigenvalue: {np.max(np.abs(np.linalg.eigvals(T))):.3f}")

## Simulate from DFM

In [ ]:
# Simulate
T_periods = 200
m = T.shape[0]

states = np.zeros((T_periods, m))
obs = np.zeros((T_periods, N))

for t in range(1, T_periods):
    eta = np.random.multivariate_normal(np.zeros(r), Q)
    states[t] = T @ states[t-1] + R @ eta
    eps = np.random.multivariate_normal(np.zeros(N), H)
    obs[t] = Z @ states[t] + eps

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(states[:, :r], linewidth=2)
axes[0].set_title('Factors')
axes[0].grid(True, alpha=0.3)

axes[1].plot(obs[:, :5], alpha=0.7)
axes[1].set_title('First 5 Observations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Successfully simulated from DFM!")